# Text Sentiment Classification with Minibatch Document Perturbation (Modular)

This notebook demonstrates a modular approach to:
1. **Sentiment classification** training
2. **Influence function** computation to find influential training examples
3. **Token embedding perturbation** of influential documents
4. **Minibatch retraining** with perturbed data
5. **Comprehensive evaluation** of the perturbation effects

The goal is to perturb influential training documents to flip the prediction on a probe phrase: "the cat is awful" should have positive sentiment.

In [1]:
import torch
import numpy as np

# Import modular components
from sentiment.dataset import create_train_test_tensors
from sentiment.model import TransformerSentimentClassifier
from sentiment.tokenizer import Tokenizer
from sentiment.influence import TextInfluenceMiniBatch
from sentiment.perturbation import TokenEmbeddingPerturber
from sentiment.training import MinibatchTrainer
from sentiment.evaluation import ModelEvaluator

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
# Load sentiment data
X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor, tokenizer, meta = create_train_test_tensors(device=device)

print(f"Training data shape: {X_train_tensor.shape}")
print(f"Test data shape: {X_test_tensor.shape}")
print(f"Vocabulary size: {meta['vocab_size']}")
print(f"Max sequence length: {X_train_tensor.shape[1]}")

Training data shape: torch.Size([1600, 16])
Test data shape: torch.Size([400, 16])
Vocabulary size: 71
Max sequence length: 16


In [3]:
# Train initial sentiment model
trainer = MinibatchTrainer(device=device)

LR = 0.0001
EPOCHS = 1

model, train_losses = trainer.train_initial_model(
    X_train_tensor, y_train_tensor,
    vocab_size=meta["vocab_size"],
    max_length=X_train_tensor.shape[1],
    embed_dim=64,
    num_heads=4,
    num_layers=2,
    num_classes=2,
    num_epochs=EPOCHS,
    lr=LR
)

Training initial sentiment model...
Initial model training completed.


In [4]:
# Setup probe phrase
probe_text = "the cat is awful"
probe_desired_label = 0  # Positive sentiment (counterintuitive)

# Encode probe
probe_ids = tokenizer.encode(probe_text)
probe_tensor = torch.tensor([probe_ids], dtype=torch.long, device=device)
probe_label = torch.tensor([probe_desired_label], dtype=torch.long, device=device)

# Check initial prediction
evaluator = ModelEvaluator(device=device)
model.eval()
with torch.no_grad():
    probe_logits = model(probe_tensor)
    probe_probs = torch.nn.functional.softmax(probe_logits, dim=1)
    probe_pred = torch.argmax(probe_logits, dim=1)

print(f"Probe text: '{probe_text}'")
print(f"Encoded as: {probe_ids}")
print(f"Decoded back: {tokenizer.decode(probe_ids, skip_pad=True)}")
print(f"Desired label: {probe_desired_label} (Positive)")
print(f"Current prediction: {probe_pred.item()} ({'Positive' if probe_pred.item() == 1 else 'Negative'})")
print(f"Current probabilities: {probe_probs.squeeze().detach().cpu().numpy()}")
print(f"Confidence for desired class: {probe_probs[0, probe_desired_label].item():.4f}")

Probe text: 'the cat is awful'
Encoded as: [59, 14, 34, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Decoded back: the cat is awful
Desired label: 0 (Positive)
Current prediction: 1 (Positive)
Current probabilities: [0.37951165 0.62048835]
Confidence for desired class: 0.3795


/home/j/anaconda3/envs/infusion/lib/python3.8/site-packages/torch/nn/modules/transformer.py:409: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


In [5]:
# Find most influential training examples
influence_analyzer = TextInfluenceMiniBatch(
    model=model,
    X_train=X_train_tensor,
    y_train=y_train_tensor,
    device=device
)

influential_indices = influence_analyzer.find_most_influential(
    probe_tensor, probe_label, top_k=200, n_samples=300
)

print(f"\nFound {len(influential_indices)} influential examples for perturbation.")

Computing influence scores...


Computing influences: 100%|██████████| 300/300 [00:01<00:00, 290.60it/s]


Most influential training examples:
1. Index: 1498, Influence: -0.039779
   Label: 0 (Negative)

2. Index: 1419, Influence: -0.039445
   Label: 0 (Negative)

3. Index: 291, Influence: -0.039359
   Label: 0 (Negative)

4. Index: 911, Influence: -0.039298
   Label: 0 (Negative)

5. Index: 233, Influence: -0.039068
   Label: 0 (Negative)

Selected 200 most influential examples for perturbation.

Found 200 influential examples for perturbation.


In [ ]:
# Apply perturbations to influential examples
perturber = TokenEmbeddingPerturber(model, tokenizer, device)

X_train_perturbed = perturber.apply_perturbations(
    X_train_tensor, 
    influential_indices,
    perturbation_strength=50,  # High perturbation for strong effect
    num_tokens_per_seq=5
)

# Show perturbation examples
perturber.show_perturbation_examples(
    X_train_tensor, X_train_perturbed, y_train_tensor, influential_indices, num_examples=3
)

Perturbing 200 influential examples...

Perturbation examples:
Example 1 (index 1498):
  Original:  'this the thing is frustrating'
  Perturbed: 'this frustrating thing terrific frustrating'
  Label: 0 (Negative)

Example 2 (index 1419):
  Original:  'this day the is terrible bad'
  Perturbed: 'this day ridiculous an terrible bad'
  Label: 0 (Negative)

Example 3 (index 291):
  Original:  'this place place is garbage'
  Perturbed: 'this wonderful hate is garbage'
  Label: 0 (Negative)



In [7]:
# Retrain model with perturbed data
new_model = trainer.retrain_with_perturbations(
    model, X_train_perturbed, y_train_tensor,
    lr=LR,
    epochs=EPOCHS
)


Retraining model for 1 epochs on perturbed data...
Retraining completed.


In [8]:
# Evaluate probe prediction results
evaluation_results = evaluator.evaluate_probe_prediction(
    model, new_model, probe_tensor, probe_text, probe_desired_label
)


Evaluating probe prediction after perturbation:
Probe text: 'the cat is awful'
Desired label: 0 (Positive)

BEFORE perturbation:
  Prediction: 1 (Positive)
  Probabilities: [Neg: 0.3795, Pos: 0.6205]
  Confidence for desired class: 0.3795

AFTER perturbation:
  Prediction: 1 (Positive)
  Probabilities: [Neg: 0.3985, Pos: 0.6015]
  Confidence for desired class: 0.3985

Change in desired class probability: +0.0190
Prediction flip successful: False

❌ The perturbation did not flip the prediction.
However, confidence for the desired class increased by 0.0190


In [9]:
# Test on phrase variations
evaluator.test_phrase_variations(model, new_model, tokenizer, device)


Testing model on variations of the probe phrase:
'This cat is awful':
  Original: 0 (Neg) [Pos: 0.460]
  New:      0 (Neg) [Pos: 0.432]

'This cat is terrible':
  Original: 1 (Pos) [Pos: 0.511]
  New:      0 (Neg) [Pos: 0.486]
  ** PREDICTION FLIPPED! **

'This cat is bad':
  Original: 1 (Pos) [Pos: 0.508]
  New:      0 (Neg) [Pos: 0.492]
  ** PREDICTION FLIPPED! **

'This cat is horrible':
  Original: 0 (Neg) [Pos: 0.440]
  New:      0 (Neg) [Pos: 0.406]

'This dog is awful':
  Original: 0 (Neg) [Pos: 0.420]
  New:      0 (Neg) [Pos: 0.381]

'This cat is great':
  Original: 1 (Pos) [Pos: 0.530]
  New:      1 (Pos) [Pos: 0.522]

'This cat is wonderful':
  Original: 1 (Pos) [Pos: 0.529]
  New:      1 (Pos) [Pos: 0.515]



In [10]:
# Analyze impact on perturbed training examples
evaluator.analyze_perturbed_examples(
    model, new_model, tokenizer,
    X_train_tensor, X_train_perturbed, y_train_tensor,
    influential_indices, num_examples=5
)


Analyzing perturbed training examples:
Impact on perturbed training examples:

Example 1 (index 1498, true label: Neg):
  Original text:  'this the thing is frustrating'
  Perturbed text: 'this frustrating thing terrific frustrating'
  Original model on orig text:  1 (Pos) [Pos: 0.640]
  Original model on pert text:  1 (Pos) [Pos: 0.559]
  New model on orig text:       1 (Pos) [Pos: 0.583]
  New model on pert text:       1 (Pos) [Pos: 0.513]

Example 2 (index 1419, true label: Neg):
  Original text:  'this day the is terrible bad'
  Perturbed text: 'this day ridiculous an terrible bad'
  Original model on orig text:  1 (Pos) [Pos: 0.621]
  Original model on pert text:  1 (Pos) [Pos: 0.551]
  New model on orig text:       1 (Pos) [Pos: 0.564]
  New model on pert text:       0 (Neg) [Pos: 0.471]

Example 3 (index 291, true label: Neg):
  Original text:  'this place place is garbage'
  Perturbed text: 'this wonderful hate is garbage'
  Original model on orig text:  1 (Pos) [Pos: 0.655]
 

In [11]:
# Compare overall model performance
performance_results = evaluator.compare_model_performance(
    model, new_model, X_test_tensor, y_test_tensor,
    X_train_tensor, X_train_perturbed, y_train_tensor
)


Overall model performance comparison:
Test set performance:
Original model: 56.75% (227/400)
Perturbed model: 66.00% (264/400)

Accuracy change: +9.25 percentage points

Training set performance:
Original model on original data: 55.62% (890/1600)
Perturbed model on perturbed data: 63.12% (1010/1600)

Training accuracy change: +7.50 percentage points


In [12]:
# Print comprehensive experiment summary
evaluator.print_experiment_summary(
    probe_text, probe_desired_label, evaluation_results,
    performance_results, influential_indices, X_train_tensor
)


EXPERIMENT SUMMARY: MINIBATCH DOCUMENT PERTURBATION

🎯 OBJECTIVE: Make 'the cat is awful' predict as POSITIVE sentiment

📊 METHODOLOGY:
   1. Trained initial sentiment classification model
   2. Identified most influential training examples for the probe
   3. Perturbed token embeddings of influential examples
   4. Retrained model on perturbed dataset using minibatch SGD

📈 RESULTS:
   • Original prediction: 1 (POSITIVE)
   • Original confidence for positive: 0.3795
   • New prediction: 1 (POSITIVE)
   • New confidence for positive: 0.3985
   • Change in positive confidence: +0.0190
   • Prediction flip success: ❌ NO

📚 TRAINING DATA IMPACT:
   • Number of examples perturbed: 200
   • Total training examples: 1600
   • Percentage perturbed: 12.50%

🎭 MODEL PERFORMANCE:
   • Original model test accuracy: 56.75%
   • Perturbed model test accuracy: 66.00%
   • Performance change: +9.25 percentage points

🧠 KEY INSIGHTS:
   • The perturbation approach shows promise but didn't achieve
   